# Navier-Stokes Flow Around a Cylinder — Documentation

This notebook documents the transient Navier-Stokes tutorial script for flow around a cylinder. The structure follows the same pattern as the reference notebook: each section explains the mathematics first, then shows the Julia code that implements it. A full runnable script is included at the end so the notebook can be executed standalone.

## 1. Governing equations

We solve the incompressible Navier-Stokes equations in strong form:

$$
\frac{\partial \mathbf{u}}{\partial t} + (\mathbf{u} \cdot \nabla)\mathbf{u} - \nu \nabla^2 \mathbf{u} + \nabla p = \mathbf{f},
$$

$$
\nabla \cdot \mathbf{u} = 0.
$$

Here $\mathbf{u} = (u_x, u_y)^T$ is velocity, $p$ is pressure, $\nu$ is the kinematic viscosity, and $\mathbf{f}$ is the body force. In this tutorial the forcing term is zero.

After multiplying the momentum equation by a test function $\mathbf{v}$ and the continuity equation by a test function $q$, then integrating over the domain, we obtain the weak form:

$$
\int_\Omega \frac{\partial \mathbf{u}}{\partial t} \cdot \mathbf{v}\, d\Omega
+ \int_\Omega [ (\mathbf{u} \cdot \nabla)\mathbf{u} ] \cdot \mathbf{v}\, d\Omega
+ \nu \int_\Omega \nabla \mathbf{u} : \nabla \mathbf{v}\, d\Omega
- \int_\Omega p\, \nabla \cdot \mathbf{v}\, d\Omega
= 0,
$$

$$
\int_\Omega q\, \nabla \cdot \mathbf{u}\, d\Omega = 0.
$$

The discrete system has the form $\mathbf{M}\dot{\mathbf{U}} + \mathbf{K}(\mathbf{U})\mathbf{U}=0$, where the convection term makes the problem nonlinear.

### Imports and parameters

The script uses Ferrite for finite elements, FerriteGmsh for geometry and mesh generation, and OrdinaryDiffEq for time stepping. We also define the fluid viscosity and the 2D spatial dimension.

In [ ]:
using Ferrite, SparseArrays, BlockArrays, LinearAlgebra, WriteVTK
using DiffEqBase
using OrdinaryDiffEqRosenbrock: Rodas5P

ν = 1.0 / 1000.0 # dynamic viscosity / kinematic viscosity used in the tutorial
dim = 2

## 2. Geometry and mesh generation

The computational domain is a channel with a circular obstacle. The rectangle defines the channel and the circle is cut out to create the cylinder. Gmsh physical groups are used to label the bottom, top, left, right, and hole boundaries so that boundary conditions can be applied robustly.

In [ ]:
using FerriteGmsh
using FerriteGmsh: Gmsh
Gmsh.initialize()
gmsh.option.set_number("General.Verbosity", 2)

rect_tag = gmsh.model.occ.add_rectangle(0, 0, 0, 1.1, 0.41)
circle_tag = gmsh.model.occ.add_circle(0.2, 0.2, 0, 0.05)
circle_curve_tag = gmsh.model.occ.add_curve_loop([circle_tag])
circle_surf_tag = gmsh.model.occ.add_plane_surface([circle_curve_tag])
gmsh.model.occ.cut([(dim, rect_tag)], [(dim, circle_surf_tag)])
gmsh.model.occ.synchronize()

bottomtag = gmsh.model.model.add_physical_group(dim - 1, [6], -1, "bottom")
lefttag   = gmsh.model.model.add_physical_group(dim - 1, [7], -1, "left")
righttag  = gmsh.model.model.add_physical_group(dim - 1, [8], -1, "right")
toptag    = gmsh.model.model.add_physical_group(dim - 1, [9], -1, "top")
holetag   = gmsh.model.model.add_physical_group(dim - 1, [5], -1, "hole")

gmsh.option.setNumber("Mesh.Algorithm", 11)
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 20)
gmsh.option.setNumber("Mesh.MeshSizeMax", 0.05)

gmsh.model.mesh.generate(dim)
grid = togrid()
Gmsh.finalize()

## 3. Finite element spaces

The tutorial uses Taylor-Hood elements, namely quadratic velocity and linear pressure. This pair is stable for incompressible flow because the velocity space is one polynomial order higher than the pressure space, which helps satisfy the inf-sup condition.

In [ ]:
ip_v = Lagrange{RefQuadrilateral, 2}()^dim
qr = QuadratureRule{RefQuadrilateral}(4)
cellvalues_v = CellValues(qr, ip_v)

ip_p = Lagrange{RefQuadrilateral, 1}()
cellvalues_p = CellValues(qr, ip_p)

dh = DofHandler(grid)
add!(dh, :v, ip_v)
add!(dh, :p, ip_p)
close!(dh)

## 4. Boundary conditions

The channel flow boundary conditions are:

$$
\mathbf{u} = \mathbf{0} \qquad \text{on the top, bottom, and cylinder boundary},
$$

$$
\mathbf{u}(0,y,t) = \left(u_{\text{in}}(t) \frac{4y(H-y)}{H^2}, 0\right)^T \qquad \text{on the inlet},
$$

where $u_{\text{in}}(t)=\min(t v_{\max}, v_{\max})$ ramps the inflow velocity from zero to the target value. The outlet is left traction-free, so no explicit Dirichlet condition is imposed there.

In [ ]:
ch = ConstraintHandler(dh)

noslip_facet_names = ["top", "bottom", "hole"]
∂Ω_noslip = union(getfacetset.((grid,), noslip_facet_names)...)
noslip_bc = Dirichlet(:v, ∂Ω_noslip, (x, t) -> Vec((0.0, 0.0)), [1, 2])
add!(ch, noslip_bc)

∂Ω_inflow = getfacetset(grid, "left")
vᵢₙ(t) = min(t * 1.5, 1.5)
parabolic_inflow_profile(x, t) = Vec((4 * vᵢₙ(t) * x[2] * (0.41 - x[2]) / 0.41^2, 0.0))
inflow_bc = Dirichlet(:v, ∂Ω_inflow, parabolic_inflow_profile, [1, 2])
add!(ch, inflow_bc)

∂Ω_free = getfacetset(grid, "right")
close!(ch)
update!(ch, 0.0)

## 5. Mass matrix

The mass matrix comes from the time derivative term:

$$
\int_\Omega \frac{\partial \mathbf{u}}{\partial t} \cdot \mathbf{v}\, d\Omega.
$$

After discretization, the local entries are

$$
M_{ij} = \int_\Omega \phi_i \cdot \phi_j\, d\Omega,
$$

and only the velocity-velocity block is non-zero because pressure has no time derivative in the incompressible formulation.

In [ ]:
function assemble_mass_matrix(cellvalues_v::CellValues, cellvalues_p::CellValues, M::SparseMatrixCSC, dh::DofHandler)
    n_basefuncs_v = getnbasefunctions(cellvalues_v)
    n_basefuncs_p = getnbasefunctions(cellvalues_p)
    n_basefuncs = n_basefuncs_v + n_basefuncs_p
    v▄, p▄ = 1, 2
    Mₑ = BlockedArray(zeros(n_basefuncs, n_basefuncs), [n_basefuncs_v, n_basefuncs_p], [n_basefuncs_v, n_basefuncs_p])

    mass_assembler = start_assemble(M)
    for cell in CellIterator(dh)
        fill!(Mₑ, 0)
        Ferrite.reinit!(cellvalues_v, cell)
        for q_point in 1:getnquadpoints(cellvalues_v)
            dΩ = getdetJdV(cellvalues_v, q_point)
            for i in 1:n_basefuncs_v
                φᵢ = shape_value(cellvalues_v, q_point, i)
                for j in 1:n_basefuncs_v
                    φⱼ = shape_value(cellvalues_v, q_point, j)
                    Mₑ[BlockIndex((v▄, v▄), (i, j))] += φᵢ ⋅ φⱼ * dΩ
                end
            end
        end
        assemble!(mass_assembler, celldofs(cell), Mₑ)
    end
    return M
end

## 6. Stokes operator

The linear viscous term produces the stiffness contribution

$$
-\nu \int_\Omega \nabla \phi_i : \nabla \phi_j\, d\Omega,
$$

and the pressure-velocity coupling terms are

$$
\int_\Omega (\nabla \cdot \phi_i)\psi_j\, d\Omega \qquad \text{and} \qquad \int_\Omega \psi_i (\nabla \cdot \phi_j)\, d\Omega.
$$

This creates the standard saddle-point matrix for incompressible flow.

In [ ]:
function assemble_stokes_matrix(cellvalues_v::CellValues, cellvalues_p::CellValues, ν, K::SparseMatrixCSC, dh::DofHandler)
    n_basefuncs_v = getnbasefunctions(cellvalues_v)
    n_basefuncs_p = getnbasefunctions(cellvalues_p)
    n_basefuncs = n_basefuncs_v + n_basefuncs_p
    v▄, p▄ = 1, 2
    Kₑ = BlockedArray(zeros(n_basefuncs, n_basefuncs), [n_basefuncs_v, n_basefuncs_p], [n_basefuncs_v, n_basefuncs_p])

    stiffness_assembler = start_assemble(K)
    for cell in CellIterator(dh)
        fill!(Kₑ, 0)
        Ferrite.reinit!(cellvalues_v, cell)
        Ferrite.reinit!(cellvalues_p, cell)
        for q_point in 1:getnquadpoints(cellvalues_v)
            dΩ = getdetJdV(cellvalues_v, q_point)
            for i in 1:n_basefuncs_v
                ∇φᵢ = shape_gradient(cellvalues_v, q_point, i)
                for j in 1:n_basefuncs_v
                    ∇φⱼ = shape_gradient(cellvalues_v, q_point, j)
                    Kₑ[BlockIndex((v▄, v▄), (i, j))] -= ν * ∇φᵢ ⊡ ∇φⱼ * dΩ
                end
            end
            for j in 1:n_basefuncs_p
                ψ = shape_value(cellvalues_p, q_point, j)
                for i in 1:n_basefuncs_v
                    divφ = shape_divergence(cellvalues_v, q_point, i)
                    Kₑ[BlockIndex((v▄, p▄), (i, j))] += (divφ * ψ) * dΩ
                    Kₑ[BlockIndex((p▄, v▄), (j, i))] += (ψ * divφ) * dΩ
                end
            end
        end
        assemble!(stiffness_assembler, celldofs(cell), Kₑ)
    end
    return K
end

## 7. Nonlinear convection term

The Navier-Stokes equations differ from Stokes flow through the convective term

$$
\int_\Omega [ (\mathbf{u}\cdot\nabla)\mathbf{u} ] \cdot \mathbf{v}\, d\Omega.
$$

This term makes the residual nonlinear and requires an iterative solver or repeated evaluation inside the time-integration residual function.

In [ ]:
function navierstokes_rhs_element!(dvₑ, vₑ, cellvalues_v)
    n_basefuncs = getnbasefunctions(cellvalues_v)
    for q_point in 1:getnquadpoints(cellvalues_v)
        dΩ = getdetJdV(cellvalues_v, q_point)
        ∇v = function_gradient(cellvalues_v, q_point, vₑ)
        v = function_value(cellvalues_v, q_point, vₑ)
        convection = v ⋅ ∇v'
        for j in 1:n_basefuncs
            φⱼ = shape_value(cellvalues_v, q_point, j)
            dvₑ[j] -= convection ⋅ φⱼ * dΩ
        end
    end
    return
end

## 8. Residual, Jacobian, and time stepping

The full residual function combines the linear Stokes operator and the nonlinear convection term. Its Jacobian is the Stokes matrix plus the linearization of the convection term. The resulting DAE is solved with `Rodas5P`, and the boundary conditions are enforced at each step by a limiter function.

In [ ]:
struct RHSparams
    K::SparseMatrixCSC
    ch::ConstraintHandler
    dh::DofHandler
    cellvalues_v::CellValues
    u::Vector
end

T = 6.0
Δt₀ = 0.001
Δt_save = 0.1

M = allocate_matrix(dh)
M = assemble_mass_matrix(cellvalues_v, cellvalues_p, M, dh)
K = allocate_matrix(dh)
K = assemble_stokes_matrix(cellvalues_v, cellvalues_p, ν, K, dh)
u₀ = zeros(ndofs(dh))
apply!(u₀, ch)
jac_sparsity = sparse(K)
apply!(M, ch)
p = RHSparams(K, ch, dh, cellvalues_v, copy(u₀))

function ferrite_limiter!(u, _, p, t)
    update!(p.ch, t)
    return apply!(u, p.ch)
end

function navierstokes!(du, u_uc, p::RHSparams, t)
    (; K, ch, dh, cellvalues_v, u) = p
    u .= u_uc
    update!(ch, t)
    apply!(u, ch)
    mul!(du, K, u)
    v_range = dof_range(dh, :v)
    n_basefuncs = getnbasefunctions(cellvalues_v)
    vₑ = zeros(n_basefuncs)
    duₑ = zeros(n_basefuncs)
    for cell in CellIterator(dh)
        Ferrite.reinit!(cellvalues_v, cell)
        v_celldofs = @view celldofs(cell)[v_range]
        vₑ .= @views u[v_celldofs]
        fill!(duₑ, 0.0)
        navierstokes_rhs_element!(duₑ, vₑ, cellvalues_v)
        assemble!(du, v_celldofs, duₑ)
    end
    return
end

function navierstokes_jac_element!(Jₑ, vₑ, cellvalues_v)
    n_basefuncs = getnbasefunctions(cellvalues_v)
    for q_point in 1:getnquadpoints(cellvalues_v)
        dΩ = getdetJdV(cellvalues_v, q_point)
        ∇v = function_gradient(cellvalues_v, q_point, vₑ)
        v = function_value(cellvalues_v, q_point, vₑ)
        for j in 1:n_basefuncs
            φⱼ = shape_value(cellvalues_v, q_point, j)
            for i in 1:n_basefuncs
                φᵢ = shape_value(cellvalues_v, q_point, i)
                ∇φᵢ = shape_gradient(cellvalues_v, q_point, i)
                Jₑ[j, i] -= (φᵢ ⋅ ∇v' + v ⋅ ∇φᵢ') ⋅ φⱼ * dΩ
            end
        end
    end
    return
end

function navierstokes_jac!(J, u_uc, p, t)
    (; K, ch, dh, cellvalues_v, u) = p
    u .= u_uc
    update!(ch, t)
    apply!(u, ch)
    nonzeros(J) .= nonzeros(K)
    assembler = start_assemble(J; fillzero = false)
    n_basefuncs = getnbasefunctions(cellvalues_v)
    Jₑ = zeros(n_basefuncs, n_basefuncs)
    vₑ = zeros(n_basefuncs)
    v_range = dof_range(dh, :v)
    for cell in CellIterator(dh)
        Ferrite.reinit!(cellvalues_v, cell)
        v_celldofs = @view celldofs(cell)[v_range]
        vₑ .= @views u[v_celldofs]
        fill!(Jₑ, 0.0)
        navierstokes_jac_element!(Jₑ, vₑ, cellvalues_v)
        assemble!(assembler, v_celldofs, Jₑ)
    end
    return apply!(J, ch)
end

rhs = ODEFunction(navierstokes!; mass_matrix = M, jac = navierstokes_jac!, jac_prototype = jac_sparsity)
problem = ODEProblem(rhs, u₀, (0.0, T), p)

struct FreeDofErrorNorm
    ch::ConstraintHandler
end
(fe_norm::FreeDofErrorNorm)(u::Union{AbstractFloat, Complex}, t) = DiffEqBase.ODE_DEFAULT_NORM(u, t)
(fe_norm::FreeDofErrorNorm)(u::AbstractArray, t) = DiffEqBase.ODE_DEFAULT_NORM(u[fe_norm.ch.free_dofs], t)

timestepper = Rodas5P(autodiff = false, step_limiter! = ferrite_limiter!)
integrator = init(
    problem, timestepper; initializealg = NoInit(), dt = Δt₀,
    adaptive = true, abstol = 1.0e-4, reltol = 1.0e-5,
    progress = true, progress_steps = 1, verbose = true,
    internalnorm = FreeDofErrorNorm(ch), d_discontinuities = [1.0]
)

## 9. Output and visualization

The solution is exported to VTK at each time step so it can be visualized in ParaView. The PVD collection file stores the time series and the individual VTU files store the mesh fields.

In [ ]:
pvd = paraview_collection("ns_tut_cyl")
for (step, (u, t)) in enumerate(intervals(integrator))
    VTKGridFile("ns_tut_cyl-$step", dh) do vtk
        write_solution(vtk, dh, u)
        pvd[t] = vtk
    end
end
vtk_save(pvd)
println("Done!")

## 10. Full runnable script

The cell below contains the full original tutorial script so the notebook can be run standalone in a Julia kernel.

In [ ]:
using Ferrite, SparseArrays, BlockArrays, LinearAlgebra, WriteVTK

using DiffEqBase
using OrdinaryDiffEqRosenbrock: Rodas5P

ν = 1.0 / 1000.0; #dynamic viscosity

using FerriteGmsh
using FerriteGmsh: Gmsh
Gmsh.initialize()
gmsh.option.set_number("General.Verbosity", 2)
dim = 2;

rect_tag = gmsh.model.occ.add_rectangle(0, 0, 0, 1.1, 0.41)
circle_tag = gmsh.model.occ.add_circle(0.2, 0.2, 0, 0.05)
circle_curve_tag = gmsh.model.occ.add_curve_loop([circle_tag])
circle_surf_tag = gmsh.model.occ.add_plane_surface([circle_curve_tag])
gmsh.model.occ.cut([(dim, rect_tag)], [(dim, circle_surf_tag)])

gmsh.model.occ.synchronize()

bottomtag = gmsh.model.model.add_physical_group(dim - 1, [6], -1, "bottom")
lefttag = gmsh.model.model.add_physical_group(dim - 1, [7], -1, "left")
righttag = gmsh.model.model.add_physical_group(dim - 1, [8], -1, "right")
toptag = gmsh.model.model.add_physical_group(dim - 1, [9], -1, "top")
holetag = gmsh.model.model.add_physical_group(dim - 1, [5], -1, "hole")

gmsh.option.setNumber("Mesh.Algorithm", 11)
gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 20)
gmsh.option.setNumber("Mesh.MeshSizeMax", 0.05)

gmsh.model.mesh.generate(dim)
grid = togrid()
Gmsh.finalize();

ip_v = Lagrange{RefQuadrilateral, 2}()^dim
qr = QuadratureRule{RefQuadrilateral}(4)
cellvalues_v = CellValues(qr, ip_v);

ip_p = Lagrange{RefQuadrilateral, 1}()
cellvalues_p = CellValues(qr, ip_p);

dh = DofHandler(grid)
add!(dh, :v, ip_v)
add!(dh, :p, ip_p)
close!(dh);

ch = ConstraintHandler(dh);

nosplip_facet_names = ["top", "bottom", "hole"];
∂Ω_noslip = union(getfacetset.((grid,), nosplip_facet_names)...);
noslip_bc = Dirichlet(:v, ∂Ω_noslip, (x, t) -> Vec((0.0, 0.0)), [1, 2])
add!(ch, noslip_bc);

∂Ω_inflow = getfacetset(grid, "left");

vᵢₙ(t) = min(t * 1.5, 1.5) #inflow velocity

parabolic_inflow_profile(x, t) = Vec((4 * vᵢₙ(t) * x[2] * (0.41 - x[2]) / 0.41^2, 0.0))
inflow_bc = Dirichlet(:v, ∂Ω_inflow, parabolic_inflow_profile, [1, 2])
add!(ch, inflow_bc);

∂Ω_free = getfacetset(grid, "right");

close!(ch)
update!(ch, 0.0);

T = 6.0
Δt₀ = 0.001
Δt_save = 0.1

function assemble_system!(K, f, dh, cvu, cvp)
    assembler = start_assemble(K, f)
    ke = zeros(ndofs_per_cell(dh), ndofs_per_cell(dh))
    fe = zeros(ndofs_per_cell(dh))
    range_u = dof_range(dh, :u)
    ndofs_u = length(range_u)
    range_p = dof_range(dh, :p)
    ndofs_p = length(range_p)
    ϕᵤ = Vector{Vec{2, Float64}}(undef, ndofs_u)
    ∇ϕᵤ = Vector{Tensor{2, 2, Float64, 4}}(undef, ndofs_u)
    divϕᵤ = Vector{Float64}(undef, ndofs_u)
    ϕₚ = Vector{Float64}(undef, ndofs_p)
    for cell in CellIterator(dh)
        Ferrite.reinit!(cvu, cell)
        Ferrite.reinit!(cvp, cell)
        ke .= 0
        fe .= 0
        for qp in 1:getnquadpoints(cvu)
            dΩ = getdetJdV(cvu, qp)
            for i in 1:ndofs_u
                ϕᵤ[i] = shape_value(cvu, qp, i)
                ∇ϕᵤ[i] = shape_gradient(cvu, qp, i)
                divϕᵤ[i] = shape_divergence(cvu, qp, i)
            end
            for i in 1:ndofs_p
                ϕₚ[i] = shape_value(cvp, qp, i)
            end
            for (i, I) in pairs(range_u), (j, J) in pairs(range_u)
                ke[I, J] += (∇ϕᵤ[i] ⊡ ∇ϕᵤ[j]) * dΩ
            end
            for (i, I) in pairs(range_u), (j, J) in pairs(range_p)
                ke[I, J] += (-divϕᵤ[i] * ϕₚ[j]) * dΩ
            end
            for (i, I) in pairs(range_p), (j, J) in pairs(range_u)
                ke[I, J] += (-divϕᵤ[j] * ϕₚ[i]) * dΩ
            end
            for (i, I) in pairs(range_u)
                x = spatial_coordinate(cvu, qp, getcoordinates(cell))
                b = exp(-100 * norm(x - Vec{2}((0.75, 0.1)))^2)
                bv = Vec{2}((b, 0.0))
                fe[I] += (ϕᵤ[i] ⋅ bv) * dΩ
            end
        end
        assemble!(assembler, celldofs(cell), ke, fe)
    end
    return K, f
end

function ferrite_limiter!(u, _, p, t)
    update!(p.ch, t)
    return apply!(u, p.ch)
end

function navierstokes_rhs_element!(dvₑ, vₑ, cellvalues_v)
    n_basefuncs = getnbasefunctions(cellvalues_v)
    for q_point in 1:getnquadpoints(cellvalues_v)
        dΩ = getdetJdV(cellvalues_v, q_point)
        ∇v = function_gradient(cellvalues_v, q_point, vₑ)
        v = function_value(cellvalues_v, q_point, vₑ)
        for j in 1:n_basefuncs
            φⱼ = shape_value(cellvalues_v, q_point, j)
            dvₑ[j] -= v ⋅ ∇v' ⋅ φⱼ * dΩ
        end
    end
    return
end

function navierstokes!(du, u_uc, p::RHSparams, t)
    (; K, ch, dh, cellvalues_v, u) = p
    u .= u_uc
    update!(ch, t)
    apply!(u, ch)
    mul!(du, K, u)
    v_range = dof_range(dh, :v)
    n_basefuncs = getnbasefunctions(cellvalues_v)
    vₑ = zeros(n_basefuncs)
    duₑ = zeros(n_basefuncs)
    for cell in CellIterator(dh)
        Ferrite.reinit!(cellvalues_v, cell)
        v_celldofs = @view celldofs(cell)[v_range]
        vₑ .= @views u[v_celldofs]
        fill!(duₑ, 0.0)
        navierstokes_rhs_element!(duₑ, vₑ, cellvalues_v)
        assemble!(du, v_celldofs, duₑ)
    end
    return
end

function navierstokes_jac_element!(Jₑ, vₑ, cellvalues_v)
    n_basefuncs = getnbasefunctions(cellvalues_v)
    for q_point in 1:getnquadpoints(cellvalues_v)
        dΩ = getdetJdV(cellvalues_v, q_point)
        ∇v = function_gradient(cellvalues_v, q_point, vₑ)
        v = function_value(cellvalues_v, q_point, vₑ)
        for j in 1:n_basefuncs
            φⱼ = shape_value(cellvalues_v, q_point, j)
            for i in 1:n_basefuncs
                φᵢ = shape_value(cellvalues_v, q_point, i)
                ∇φᵢ = shape_gradient(cellvalues_v, q_point, i)
                Jₑ[j, i] -= (φᵢ ⋅ ∇v' + v ⋅ ∇φᵢ') ⋅ φⱼ * dΩ
            end
        end
    end
    return
end

function navierstokes_jac!(J, u_uc, p, t)
    (; K, ch, dh, cellvalues_v, u) = p
    u .= u_uc
    update!(ch, t)
    apply!(u, ch)
    nonzeros(J) .= nonzeros(K)
    assembler = start_assemble(J; fillzero = false)
    n_basefuncs = getnbasefunctions(cellvalues_v)
    Jₑ = zeros(n_basefuncs, n_basefuncs)
    vₑ = zeros(n_basefuncs)
    v_range = dof_range(dh, :v)
    for cell in CellIterator(dh)
        Ferrite.reinit!(cellvalues_v, cell)
        v_celldofs = @view celldofs(cell)[v_range]
        vₑ .= @views u[v_celldofs]
        fill!(Jₑ, 0.0)
        navierstokes_jac_element!(Jₑ, vₑ, cellvalues_v)
        assemble!(assembler, v_celldofs, Jₑ)
    end
    return apply!(J, ch)
end

jac_sparsity = sparse(K)

struct FreeDofErrorNorm
    ch::ConstraintHandler
end
(fe_norm::FreeDofErrorNorm)(u::Union{AbstractFloat, Complex}, t) = DiffEqBase.ODE_DEFAULT_NORM(u, t)
(fe_norm::FreeDofErrorNorm)(u::AbstractArray, t) = DiffEqBase.ODE_DEFAULT_NORM(u[fe_norm.ch.free_dofs], t)

rhs = ODEFunction(navierstokes!; mass_matrix = M, jac = navierstokes_jac!, jac_prototype = jac_sparsity)
problem = ODEProblem(rhs, u₀, (0.0, T), p)

timestepper = Rodas5P(autodiff = false, step_limiter! = ferrite_limiter!)
integrator = init(
    problem, timestepper;
    initializealg = NoInit(),
    dt = Δt₀,
    adaptive = true,
    abstol = 1.0e-4,
    reltol = 1.0e-5,
    progress = true,
    progress_steps = 1,
    verbose = true,
    internalnorm = FreeDofErrorNorm(ch),
    d_discontinuities = [1.0]
)

pvd = paraview_collection("ns_tut_cyl")
for (step, (u, t)) in enumerate(intervals(integrator))
    VTKGridFile("ns_tut_cyl-$step", dh) do vtk
        write_solution(vtk, dh, u)
        pvd[t] = vtk
    end
end
vtk_save(pvd)

println("Done!")

---

Notes:
- This notebook is meant to be run in a Julia kernel with IJulia installed.
- The full script cell duplicates the sectioned cells so the notebook can be executed from top to bottom without needing the original `.jl` file.
- The outlet is intentionally left traction-free, matching the original tutorial setup.